In [2]:
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [3]:
import pandas as pd
import requests
from io import StringIO

url = "https://www3.cde.ca.gov/demo-downloads/sped/spedps2425.txt"

r = requests.get(url)
r.raise_for_status()
text = r.content.decode("cp1252", errors="replace")

In [4]:
df = pd.read_csv(
    StringIO(text),
    sep="\t",
    dtype={
        "Academic Year": "string",
        "Aggregate Level": "string",
        "County Code": "string",
        "District Code": "string",
        "School Code": "string",
        "County Name": "string",
        "District Name": "string",
        "School Name": "string",
        "Charter School": "string",
        "ReportingCategory": "string"
    },
    keep_default_na=False
)

df.columns = df.columns.str.strip()

string_cols = [
    "Academic Year", "Aggregate Level", "County Code", "District Code", "School Code",
    "County Name", "District Name", "School Name", "Charter School", "ReportingCategory"
]

for col in string_cols:
    df[col] = df[col].str.strip()

# keep only school-level rows
sped_df = df[df["Aggregate Level"] == "S"].copy()

# school-level should only be TA, but keep this filter for safety
sped_df = sped_df[sped_df["ReportingCategory"] == "TA"].copy()

# format codes
sped_df["School Code"] = sped_df["School Code"].astype(int).astype(str).str.zfill(7)

# filter to your school list
sped_df = sped_df[sped_df["School Code"].isin(school_codes_list)].copy()
sped_df = sped_df.reset_index(drop=True)

In [9]:
sped_df = sped_df[['School Code', 'SPED_ENR_N']]

In [10]:
sped_df

,School Code,SPED_ENR_N
0,0130625,19
1,0106401,19
2,0130229,235
3,0130450,92
4,0131177,357
...,...,...
1379,5738802,169
1380,5830047,5
1381,5830013,179
1382,5835202,151


In [ ]:
sped_df.to_csv('../cleaned_data/sped_data.csv')